### Product Demand Forecasting

#### 1. Business Understanding

The PT Frozen Foods platform now incorporates a Machine Learning workflow focused on product demand forecasting.

The objective of this notebook is to analyze historical product sales behavior and prepare the foundation for a forecasting model capable of estimating future product demand.

The forecasting use case supports business decisions related to:

- inventory planning
- purchasing decisions
- supplier planning
- logistics preparation
- commercial strategy
- marketing actions
- financial planning

The selected forecasting target is product quantity sold.

Official forecasting grain:

    data_pedido + produto_id

Official target:

    quantity_sold

Forecast horizon:

    90 days

This notebook is exploratory and is not intended to be used as a production processing artifact.

In [0]:
# ========================================
# ENVIRONMENT VALIDATION
# ========================================

import sys

print("=" * 80)
print("ENVIRONMENT VALIDATION")
print("=" * 80)

print(f"Python version: {sys.version}")
print(f"Spark version:  {spark.version}")

# ========================================
# Optional libraries
# ========================================

OPTIONAL_LIBRARIES = [
    "sklearn",
    "xgboost",
    "lightgbm",
    "mlflow",
    "pandas",
    "numpy",
    "plotly"
]

for lib in OPTIONAL_LIBRARIES:

    try:
        module = __import__(lib)

        version = getattr(module, "__version__", "unknown")

        print(f"[AVAILABLE] {lib} - version: {version}")

    except Exception as e:

        print(f"[NOT AVAILABLE] {lib} - {str(e)}")

print("=" * 80)
print("ENVIRONMENT VALIDATION COMPLETED")
print("=" * 80)

In [0]:
# ========================================
# 0. CONFIGURATION
# ========================================

import pandas as pd
import numpy as np

import pyspark.sql.functions as F
from pyspark.sql.window import Window

import matplotlib.pyplot as plt
import plotly.express as px

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "gold"
TARGET_SCHEMA = "gold"

DOMAIN = "ml"
USE_CASE = "product_demand_forecasting"

SOURCE_DATASET = "analytics_sales_overview"
SOURCE_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{SOURCE_DATASET}"

FORECAST_GRAIN = [
    "data_pedido",
    "produto_id"
]

TARGET_COLUMN = "quantity_sold"

FORECAST_HORIZON_DAYS = 90

print("=" * 80)
print("STARTING ML EXPLORATORY NOTEBOOK: product_demand_forecasting")
print("=" * 80)

print(f"Source table:          {SOURCE_TABLE}")
print(f"Forecast grain:        {FORECAST_GRAIN}")
print(f"Target column:         {TARGET_COLUMN}")
print(f"Forecast horizon days: {FORECAST_HORIZON_DAYS}")

In [0]:
# ========================================
# 1. LOAD DATASET
# ========================================

print("=" * 80)
print("LOADING SOURCE DATASET")
print("=" * 80)

source_df = spark.table(SOURCE_TABLE)

print(f"[INFO] Source table loaded successfully: {SOURCE_TABLE}")

print("[INFO] Dataset schema:")
source_df.printSchema()

print("=" * 80)
print("SOURCE DATASET PREVIEW")
print("=" * 80)

display(
    source_df.limit(10)
)

In [0]:
# ========================================
# 2. INITIAL DATASET VALIDATION
# ========================================

print("=" * 80)
print("INITIAL DATASET VALIDATION")
print("=" * 80)

total_rows = source_df.count()

date_stats = (
    source_df
    .agg(
        F.min("data_pedido").alias("min_data_pedido"),
        F.max("data_pedido").alias("max_data_pedido"),
        F.countDistinct("data_pedido").alias("distinct_order_dates"),
        F.countDistinct("produto_id").alias("distinct_products")
    )
    .collect()[0]
)

grain_rows = (
    source_df
    .select(FORECAST_GRAIN)
    .distinct()
    .count()
)

duplicate_grain_rows = total_rows - grain_rows

null_check = (
    source_df
    .select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in FORECAST_GRAIN + [TARGET_COLUMN]
    ])
    .collect()[0]
    .asDict()
)

print(f"Total rows:              {total_rows:,}")
print(f"Min order date:          {date_stats['min_data_pedido']}")
print(f"Max order date:          {date_stats['max_data_pedido']}")
print(f"Distinct order dates:    {date_stats['distinct_order_dates']:,}")
print(f"Distinct products:       {date_stats['distinct_products']:,}")
print(f"Distinct grain rows:     {grain_rows:,}")
print(f"Duplicate grain rows:    {duplicate_grain_rows:,}")
print(f"Null check:              {null_check}")

if any(value != 0 for value in null_check.values()):
    raise ValueError("Null values found in critical forecasting columns.")

print("[INFO] Initial dataset validation completed.")

In [0]:
# ========================================
# 3. AGGREGATE FORECASTING DATASET
# ========================================

print("=" * 80)
print("AGGREGATING FORECASTING DATASET")
print("=" * 80)

forecast_base_df = (
    source_df
    .groupBy(
        "data_pedido",
        "produto_id"
    )
    .agg(
        F.sum("quantity_sold").alias("quantity_sold"),
        F.sum("net_sales_amount").alias("net_sales_amount"),
        F.sum("gross_sales_amount").alias("gross_sales_amount"),
        F.sum("total_cost_amount").alias("total_cost_amount"),
        F.sum("gross_margin_amount").alias("gross_margin_amount"),
        F.sum("order_count").alias("order_count"),
        F.sum("line_count").alias("line_count"),

        F.first("produto_nome", ignorenulls=True).alias("produto_nome"),
        F.first("categoria", ignorenulls=True).alias("categoria"),
        F.first("marca", ignorenulls=True).alias("marca"),
        F.first("status_produto", ignorenulls=True).alias("status_produto"),

        F.countDistinct("cliente_id").alias("distinct_customers"),
        F.countDistinct("canal_id").alias("distinct_channels"),
        F.countDistinct("vendedor_id").alias("distinct_salespersons")
    )
)

forecast_base_df = (
    forecast_base_df
    .withColumn("avg_net_unit_price", F.col("net_sales_amount") / F.col("quantity_sold"))
    .withColumn("avg_gross_unit_price", F.col("gross_sales_amount") / F.col("quantity_sold"))
    .withColumn("avg_unit_cost", F.col("total_cost_amount") / F.col("quantity_sold"))
    .withColumn("avg_unit_margin", F.col("gross_margin_amount") / F.col("quantity_sold"))
)

total_rows = forecast_base_df.count()

distinct_grain_rows = (
    forecast_base_df
    .select(FORECAST_GRAIN)
    .distinct()
    .count()
)

duplicate_grain_rows = total_rows - distinct_grain_rows

date_stats = (
    forecast_base_df
    .agg(
        F.min("data_pedido").alias("min_data_pedido"),
        F.max("data_pedido").alias("max_data_pedido"),
        F.countDistinct("data_pedido").alias("distinct_dates"),
        F.countDistinct("produto_id").alias("distinct_products"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.sum("net_sales_amount").alias("total_net_sales_amount"),
        F.avg("quantity_sold").alias("avg_daily_product_quantity"),
        F.max("quantity_sold").alias("max_daily_product_quantity")
    )
    .collect()[0]
)

null_check = (
    forecast_base_df
    .select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in FORECAST_GRAIN + [
            "quantity_sold",
            "net_sales_amount",
            "produto_nome",
            "categoria",
            "marca",
            "status_produto"
        ]
    ])
    .collect()[0]
    .asDict()
)

print(f"Forecast base rows:              {total_rows:,}")
print(f"Distinct grain rows:             {distinct_grain_rows:,}")
print(f"Duplicate grain rows:            {duplicate_grain_rows:,}")
print(f"Min order date:                  {date_stats['min_data_pedido']}")
print(f"Max order date:                  {date_stats['max_data_pedido']}")
print(f"Distinct dates:                  {date_stats['distinct_dates']:,}")
print(f"Distinct products:               {date_stats['distinct_products']:,}")
print(f"Total quantity sold:             {date_stats['total_quantity_sold']:,.2f}")
print(f"Total net sales amount:          {date_stats['total_net_sales_amount']:,.2f}")
print(f"Avg daily product quantity:      {date_stats['avg_daily_product_quantity']:,.2f}")
print(f"Max daily product quantity:      {date_stats['max_daily_product_quantity']:,.2f}")
print(f"Null check:                      {null_check}")

if duplicate_grain_rows != 0:
    raise ValueError("Duplicate grain records found after forecasting aggregation.")

if any(value != 0 for value in null_check.values()):
    raise ValueError("Null values found in critical forecasting columns.")

print("[INFO] Forecasting dataset aggregated and validated successfully.")

display(
    forecast_base_df
    .orderBy("data_pedido", "produto_id")
    .limit(20)
)

In [0]:
# ========================================
# 4. PRODUCT TEMPORAL COVERAGE ANALYSIS
# ========================================

print("=" * 80)
print("PRODUCT TEMPORAL COVERAGE ANALYSIS")
print("=" * 80)

product_coverage_df = (
    forecast_base_df
    .groupBy(
        "produto_id",
        "produto_nome",
        "categoria",
        "marca",
        "status_produto"
    )
    .agg(
        F.min("data_pedido").alias("first_sale_date"),
        F.max("data_pedido").alias("last_sale_date"),
        F.countDistinct("data_pedido").alias("active_days"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.avg("quantity_sold").alias("avg_quantity_sold"),
        F.max("quantity_sold").alias("max_quantity_sold"),
        F.sum("net_sales_amount").alias("total_net_sales_amount")
    )
)

total_dataset_days = (
    forecast_base_df
    .select(F.countDistinct("data_pedido").alias("total_days"))
    .collect()[0]["total_days"]
)

product_coverage_df = (
    product_coverage_df
    .withColumn(
        "coverage_ratio",
        F.col("active_days") / F.lit(total_dataset_days)
    )
    .withColumn(
        "inactive_days",
        F.lit(total_dataset_days) - F.col("active_days")
    )
)

print(f"Total dataset days: {total_dataset_days:,}")

display(
    product_coverage_df
    .orderBy(F.col("coverage_ratio").desc())
)

In [0]:
# ========================================
# 5. GLOBAL DEMAND TIME SERIES
# ========================================

print("=" * 80)
print("GLOBAL DEMAND TIME SERIES")
print("=" * 80)

global_daily_df = (
    forecast_base_df
    .groupBy("data_pedido")
    .agg(
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.sum("net_sales_amount").alias("total_net_sales_amount"),
        F.countDistinct("produto_id").alias("active_products")
    )
    .orderBy("data_pedido")
)

global_daily_pdf = global_daily_df.toPandas()

print(f"Total daily records: {len(global_daily_pdf):,}")

display(global_daily_df)

fig = px.line(
    global_daily_pdf,
    x="data_pedido",
    y="total_quantity_sold",
    title="Global Daily Product Demand"
)

fig.show()

In [0]:
# ========================================
# 6. SEASONALITY ANALYSIS
# ========================================

print("=" * 80)
print("SEASONALITY ANALYSIS")
print("=" * 80)

seasonality_df = (
    forecast_base_df
    .withColumn("calendar_month", F.month("data_pedido"))
    .withColumn("day_of_week", F.dayofweek("data_pedido"))
    .groupBy("calendar_month", "day_of_week")
    .agg(
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.avg("quantity_sold").alias("avg_quantity_sold"),
        F.countDistinct("data_pedido").alias("active_days"),
        F.countDistinct("produto_id").alias("active_products")
    )
    .orderBy("calendar_month", "day_of_week")
)

seasonality_pdf = seasonality_df.toPandas()

display(seasonality_df)

fig = px.bar(
    seasonality_pdf,
    x="calendar_month",
    y="total_quantity_sold",
    color="day_of_week",
    title="Product Demand Seasonality by Month and Day of Week",
    barmode="group"
)

fig.show()

In [0]:
# ========================================
# 7. CREATE COMPLETE PRODUCT-DATE TIME SERIES
# ========================================

print("=" * 80)
print("CREATING COMPLETE PRODUCT-DATE TIME SERIES")
print("=" * 80)

# ========================================
# Create complete calendar
# ========================================

calendar_df = (
    forecast_base_df
    .select("data_pedido")
    .distinct()
)

# ========================================
# Create product dimension
# ========================================

products_df = (
    forecast_base_df
    .select(
        "produto_id",
        "produto_nome",
        "categoria",
        "marca",
        "status_produto"
    )
    .distinct()
)

# ========================================
# Create complete date-product combinations
# ========================================

complete_series_df = (
    calendar_df
    .crossJoin(products_df)
)

# ========================================
# Join existing sales data
# ========================================

complete_series_df = (
    complete_series_df
    .join(
        forecast_base_df,
        on=[
            "data_pedido",
            "produto_id",
            "produto_nome",
            "categoria",
            "marca",
            "status_produto"
        ],
        how="left"
    )
)

# ========================================
# Fill missing values
# ========================================

metric_columns = [
    "quantity_sold",
    "net_sales_amount",
    "gross_sales_amount",
    "total_cost_amount",
    "gross_margin_amount",
    "order_count",
    "line_count",
    "distinct_customers",
    "distinct_channels",
    "distinct_salespersons"
]

complete_series_df = complete_series_df.fillna(0, subset=metric_columns)

# ========================================
# Validation
# ========================================

total_rows = complete_series_df.count()

distinct_dates = (
    complete_series_df
    .select(F.countDistinct("data_pedido").alias("distinct_dates"))
    .collect()[0]["distinct_dates"]
)

distinct_products = (
    complete_series_df
    .select(F.countDistinct("produto_id").alias("distinct_products"))
    .collect()[0]["distinct_products"]
)

expected_rows = distinct_dates * distinct_products

duplicate_grain_rows = (
    total_rows
    -
    complete_series_df
    .select("data_pedido", "produto_id")
    .distinct()
    .count()
)

zero_demand_rows = (
    complete_series_df
    .filter(F.col("quantity_sold") == 0)
    .count()
)

print(f"Distinct dates:                 {distinct_dates:,}")
print(f"Distinct products:              {distinct_products:,}")
print(f"Expected rows:                  {expected_rows:,}")
print(f"Actual rows:                    {total_rows:,}")
print(f"Duplicate grain rows:           {duplicate_grain_rows:,}")
print(f"Zero demand rows created:       {zero_demand_rows:,}")

if total_rows != expected_rows:
    raise ValueError("Complete series row count mismatch.")

if duplicate_grain_rows != 0:
    raise ValueError("Duplicate grain rows found in complete series.")

print("[INFO] Complete forecasting time series created successfully.")

display(
    complete_series_df
    .orderBy("produto_id", "data_pedido")
    .limit(50)
)

In [0]:
# ========================================
# 8. ZERO DEMAND AND SPARSITY ANALYSIS
# ========================================

print("=" * 80)
print("ZERO DEMAND AND SPARSITY ANALYSIS")
print("=" * 80)

sparsity_df = (
    complete_series_df
    .groupBy(
        "produto_id",
        "produto_nome",
        "categoria",
        "marca",
        "status_produto"
    )
    .agg(
        F.count("*").alias("total_days"),
        F.sum(F.when(F.col("quantity_sold") > 0, 1).otherwise(0)).alias("active_sales_days"),
        F.sum(F.when(F.col("quantity_sold") == 0, 1).otherwise(0)).alias("zero_sales_days"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.avg("quantity_sold").alias("avg_daily_quantity_including_zeros"),
        F.avg(F.when(F.col("quantity_sold") > 0, F.col("quantity_sold"))).alias("avg_quantity_when_sold"),
        F.max("quantity_sold").alias("max_daily_quantity")
    )
    .withColumn(
        "zero_sales_ratio",
        F.col("zero_sales_days") / F.col("total_days")
    )
    .withColumn(
        "active_sales_ratio",
        F.col("active_sales_days") / F.col("total_days")
    )
    .orderBy(F.col("zero_sales_ratio").desc())
)

display(sparsity_df)

sparsity_summary = (
    sparsity_df
    .agg(
        F.avg("zero_sales_ratio").alias("avg_zero_sales_ratio"),
        F.min("zero_sales_ratio").alias("min_zero_sales_ratio"),
        F.max("zero_sales_ratio").alias("max_zero_sales_ratio"),
        F.count(F.when(F.col("zero_sales_ratio") >= 0.50, True)).alias("products_above_50pct_zero_days"),
        F.count(F.when(F.col("zero_sales_ratio") >= 0.70, True)).alias("products_above_70pct_zero_days")
    )
    .collect()[0]
)

print(f"Average zero sales ratio:        {sparsity_summary['avg_zero_sales_ratio']:.4f}")
print(f"Minimum zero sales ratio:        {sparsity_summary['min_zero_sales_ratio']:.4f}")
print(f"Maximum zero sales ratio:        {sparsity_summary['max_zero_sales_ratio']:.4f}")
print(f"Products >= 50% zero days:       {sparsity_summary['products_above_50pct_zero_days']}")
print(f"Products >= 70% zero days:       {sparsity_summary['products_above_70pct_zero_days']}")

print("[INFO] Zero demand and sparsity analysis completed.")

In [0]:
# ========================================
# 9. CREATE ELIGIBLE FORECASTING DATASET
# ========================================

print("=" * 80)
print("CREATING ELIGIBLE FORECASTING DATASET")
print("=" * 80)

MIN_ACTIVE_RATIO = 0.50

eligible_products_df = (
    sparsity_df
    .filter(F.col("active_sales_ratio") >= MIN_ACTIVE_RATIO)
    .select("produto_id")
)

eligible_forecast_df = (
    complete_series_df
    .join(
        eligible_products_df,
        on="produto_id",
        how="inner"
    )
)

eligible_product_count = (
    eligible_products_df
    .count()
)

eligible_rows = (
    eligible_forecast_df
    .count()
)

excluded_product_count = (
    products_df.count()
    -
    eligible_product_count
)

print(f"Minimum active ratio:            {MIN_ACTIVE_RATIO:.2f}")
print(f"Eligible products:               {eligible_product_count}")
print(f"Excluded products:               {excluded_product_count}")
print(f"Eligible forecasting rows:       {eligible_rows:,}")

print("[INFO] Eligible forecasting dataset created successfully.")

display(
    eligible_forecast_df
    .orderBy("produto_id", "data_pedido")
    .limit(50)
)

In [0]:
# ========================================
# 10. CREATE CALENDAR FEATURES
# ========================================

print("=" * 80)
print("CREATING CALENDAR FEATURES")
print("=" * 80)

feature_df = eligible_forecast_df

feature_df = (
    feature_df
    .withColumn("calendar_year", F.year("data_pedido"))
    .withColumn("calendar_month", F.month("data_pedido"))
    .withColumn("calendar_day", F.dayofmonth("data_pedido"))
    .withColumn("calendar_quarter", F.quarter("data_pedido"))
    .withColumn("week_of_year", F.weekofyear("data_pedido"))
    .withColumn("day_of_week", F.dayofweek("data_pedido"))
    .withColumn(
        "is_weekend",
        F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0)
    )
)

print("[INFO] Calendar features created successfully.")

display(
    feature_df
    .select(
        "data_pedido",
        "produto_id",
        "quantity_sold",
        "calendar_year",
        "calendar_month",
        "calendar_day",
        "calendar_quarter",
        "week_of_year",
        "day_of_week",
        "is_weekend"
    )
    .orderBy("produto_id", "data_pedido")
    .limit(50)
)

In [0]:
# ========================================
# 11. CREATE LAG FEATURES
# ========================================

print("=" * 80)
print("CREATING LAG FEATURES")
print("=" * 80)

window_spec = (
    Window
    .partitionBy("produto_id")
    .orderBy("data_pedido")
)

LAG_PERIODS = [1, 7, 30, 90, 365]

for lag_period in LAG_PERIODS:

    feature_df = (
        feature_df
        .withColumn(
            f"sales_lag_{lag_period}",
            F.lag("quantity_sold", lag_period).over(window_spec)
        )
    )

print(f"[INFO] Lag features created: {LAG_PERIODS}")

display(
    feature_df
    .select(
        "data_pedido",
        "produto_id",
        "quantity_sold",
        "sales_lag_1",
        "sales_lag_7",
        "sales_lag_30",
        "sales_lag_90",
        "sales_lag_365"
    )
    .orderBy("produto_id", "data_pedido")
    .limit(100)
)

In [0]:
# ========================================
# 12. CREATE ROLLING FEATURES
# ========================================

print("=" * 80)
print("CREATING ROLLING FEATURES")
print("=" * 80)

ROLLING_WINDOWS = [7, 30, 90]

for rolling_period in ROLLING_WINDOWS:

    rolling_window = (
        Window
        .partitionBy("produto_id")
        .orderBy("data_pedido")
        .rowsBetween(-rolling_period, -1)
    )

    feature_df = (
        feature_df
        .withColumn(
            f"rolling_avg_{rolling_period}",
            F.avg("quantity_sold").over(rolling_window)
        )
    )

print(f"[INFO] Rolling average features created: {ROLLING_WINDOWS}")

display(
    feature_df
    .select(
        "data_pedido",
        "produto_id",
        "quantity_sold",
        "rolling_avg_7",
        "rolling_avg_30",
        "rolling_avg_90"
    )
    .orderBy("produto_id", "data_pedido")
    .limit(100)
)

In [0]:
# ========================================
# 13. FEATURE NULL VALIDATION
# ========================================

print("=" * 80)
print("FEATURE NULL VALIDATION")
print("=" * 80)

FEATURE_COLUMNS = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_30",
    "sales_lag_90",
    "sales_lag_365",
    "rolling_avg_7",
    "rolling_avg_30",
    "rolling_avg_90"
]

feature_null_check = (
    feature_df
    .select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in FEATURE_COLUMNS
    ])
    .collect()[0]
    .asDict()
)

total_rows = feature_df.count()

print(f"Total feature rows: {total_rows:,}")

for column_name, null_count in feature_null_check.items():
    null_ratio = null_count / total_rows
    print(f"{column_name}: {null_count:,} nulls ({null_ratio:.2%})")

print("[INFO] Feature null validation completed.")

In [0]:
# ========================================
# 14. CREATE CLEAN FORECASTING TRAINING DATASET
# ========================================

print("=" * 80)
print("CREATING CLEAN FORECASTING TRAINING DATASET")
print("=" * 80)

initial_rows = feature_df.count()

training_df = (
    feature_df
    .filter(
        F.col("sales_lag_365").isNotNull()
    )
)

final_rows = training_df.count()

removed_rows = initial_rows - final_rows

removed_ratio = removed_rows / initial_rows

print(f"Initial rows:                 {initial_rows:,}")
print(f"Final training rows:          {final_rows:,}")
print(f"Removed rows:                 {removed_rows:,}")
print(f"Removed ratio:                {removed_ratio:.2%}")

print("[INFO] Clean forecasting training dataset created successfully.")

display(
    training_df
    .orderBy("produto_id", "data_pedido")
    .limit(100)
)

In [0]:
# ========================================
# 15. TEMPORAL TRAIN VALIDATION TEST SPLIT
# ========================================

print("=" * 80)
print("TEMPORAL TRAIN VALIDATION TEST SPLIT")
print("=" * 80)

MAX_DATE = (
    training_df
    .agg(F.max("data_pedido").alias("max_date"))
    .collect()[0]["max_date"]
)

VALIDATION_DAYS = 90
TEST_DAYS = 90

test_start_date = (
    pd.Timestamp(MAX_DATE)
    -
    pd.Timedelta(days=TEST_DAYS - 1)
)

validation_start_date = (
    test_start_date
    -
    pd.Timedelta(days=VALIDATION_DAYS)
)

print(f"Max dataset date:              {MAX_DATE}")
print(f"Validation horizon days:       {VALIDATION_DAYS}")
print(f"Test horizon days:             {TEST_DAYS}")

print(f"Validation start date:         {validation_start_date.date()}")
print(f"Test start date:               {test_start_date.date()}")

# ========================================
# Train dataset
# ========================================

train_df = (
    training_df
    .filter(
        F.col("data_pedido") < F.lit(validation_start_date)
    )
)

# ========================================
# Validation dataset
# ========================================

validation_df = (
    training_df
    .filter(
        (F.col("data_pedido") >= F.lit(validation_start_date))
        &
        (F.col("data_pedido") < F.lit(test_start_date))
    )
)

# ========================================
# Test dataset
# ========================================

test_df = (
    training_df
    .filter(
        F.col("data_pedido") >= F.lit(test_start_date)
    )
)

# ========================================
# Validation
# ========================================

train_rows = train_df.count()
validation_rows = validation_df.count()
test_rows = test_df.count()

print(f"Train rows:                    {train_rows:,}")
print(f"Validation rows:               {validation_rows:,}")
print(f"Test rows:                     {test_rows:,}")

print("[INFO] Temporal train/validation/test split completed successfully.")

In [0]:
# ========================================
# 16. CREATE NAIVE FORECAST BASELINE
# ========================================

print("=" * 80)
print("CREATING NAIVE FORECAST BASELINE")
print("=" * 80)

naive_baseline_df = (
    validation_df
    .select(
        "data_pedido",
        "produto_id",
        "quantity_sold",
        "sales_lag_1"
    )
    .withColumnRenamed("quantity_sold", "actual_quantity")
    .withColumnRenamed("sales_lag_1", "predicted_quantity")
)

print("[INFO] Naive forecast baseline created successfully.")

display(
    naive_baseline_df
    .orderBy("produto_id", "data_pedido")
    .limit(100)
)

In [0]:
# ========================================
# 17. EVALUATE NAIVE FORECAST BASELINE
# ========================================

print("=" * 80)
print("EVALUATING NAIVE FORECAST BASELINE")
print("=" * 80)

baseline_metrics_df = (
    naive_baseline_df
    .withColumn(
        "absolute_error",
        F.abs(
            F.col("actual_quantity")
            -
            F.col("predicted_quantity")
        )
    )
    .withColumn(
        "squared_error",
        F.pow(
            F.col("actual_quantity")
            -
            F.col("predicted_quantity"),
            2
        )
    )
)

metrics = (
    baseline_metrics_df
    .agg(
        F.avg("absolute_error").alias("mae"),
        F.sqrt(F.avg("squared_error")).alias("rmse")
    )
    .collect()[0]
)

mae = metrics["mae"]
rmse = metrics["rmse"]

print(f"MAE:   {mae:.4f}")
print(f"RMSE:  {rmse:.4f}")

print("[INFO] Naive forecast baseline evaluation completed.")

In [0]:
# ========================================
# 18. BENCHMARK FORECASTING MODELS
# ========================================

print("=" * 80)
print("BENCHMARKING FORECASTING MODELS ON VALIDATION SET")
print("=" * 80)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ========================================
# Feature definition
# ========================================

NUMERIC_FEATURES = [
    "calendar_year",
    "calendar_month",
    "calendar_day",
    "calendar_quarter",
    "week_of_year",
    "day_of_week",
    "is_weekend",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_30",
    "sales_lag_90",
    "sales_lag_365",
    "rolling_avg_7",
    "rolling_avg_30",
    "rolling_avg_90"
]

CATEGORICAL_FEATURES = [
    "produto_id",
    "categoria",
    "marca",
    "status_produto"
]

MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# ========================================
# Convert train and validation datasets to Pandas
# ========================================

train_pdf = (
    train_df
    .select(MODEL_FEATURES + [TARGET_COLUMN])
    .toPandas()
)

validation_pdf = (
    validation_df
    .select(MODEL_FEATURES + [TARGET_COLUMN])
    .toPandas()
)

X_train = train_pdf[MODEL_FEATURES]
y_train = train_pdf[TARGET_COLUMN]

X_validation = validation_pdf[MODEL_FEATURES]
y_validation = validation_pdf[TARGET_COLUMN]

print(f"Train shape:       {X_train.shape}")
print(f"Validation shape:  {X_validation.shape}")

# ========================================
# Preprocessing
# ========================================

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("numeric", "passthrough", NUMERIC_FEATURES)
    ]
)

# ========================================
# Models
# ========================================

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=300,
        max_depth=-1,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

# ========================================
# Baseline models
# ========================================

results = []

baseline_predictions = {
    "Naive Lag 1": validation_pdf["sales_lag_1"],
    "Moving Average 7": validation_pdf["rolling_avg_7"],
    "Moving Average 30": validation_pdf["rolling_avg_30"],
    "Moving Average 90": validation_pdf["rolling_avg_90"]
}

for model_name, predictions in baseline_predictions.items():

    predictions = np.maximum(predictions, 0)

    mae = mean_absolute_error(y_validation, predictions)
    rmse = mean_squared_error(y_validation, predictions, squared=False)

    results.append({
        "model_name": model_name,
        "model_type": "baseline",
        "mae": mae,
        "rmse": rmse
    })

# ========================================
# ML models
# ========================================

trained_models = {}

for model_name, model in models.items():

    print(f"[INFO] Training model: {model_name}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_validation)
    predictions = np.maximum(predictions, 0)

    mae = mean_absolute_error(y_validation, predictions)
    rmse = mean_squared_error(y_validation, predictions, squared=False)

    results.append({
        "model_name": model_name,
        "model_type": "machine_learning",
        "mae": mae,
        "rmse": rmse
    })

    trained_models[model_name] = pipeline

    print(f"[INFO] {model_name} completed. MAE: {mae:.4f} | RMSE: {rmse:.4f}")

# ========================================
# Results
# ========================================

model_results_pdf = (
    pd.DataFrame(results)
    .sort_values("mae")
    .reset_index(drop=True)
)

display(model_results_pdf)

best_model_name = model_results_pdf.iloc[0]["model_name"]

print("=" * 80)
print("MODEL BENCHMARK COMPLETED")
print("=" * 80)
print(f"Best validation model: {best_model_name}")      